In [2]:
import geopandas as gpd
import pandas as pd
import folium

In [3]:
berk_parcels = gpd.read_parquet("../data/processed/berk_parcels.parquet")
oak_parcels = gpd.read_parquet("../data/processed/oak_scaled_parcels.parquet")
sf_parcels = gpd.read_parquet("../data/processed/sf_sb79_parcels_scaled.parquet")

In [7]:
def prep_gdf(gdf, col_name, city_name):
  # Reproject, reset row index, and drop any pre-existing duplicate columns
  gdf_clean = gdf.to_crs(epsg=4326).reset_index(drop=True)
  gdf_clean = gdf_clean.loc[:, ~gdf_clean.columns.duplicated()]

  # Build a fresh GeoDataFrame to guarantee identical, unique column names
  return gpd.GeoDataFrame(
      {
          "city": city_name,
          "scale_change": gdf_clean[col_name],
      },
      geometry=gdf_clean.geometry,
      crs="EPSG:4326",
  )


berk = prep_gdf(berk_parcels, "net_scale_increase", "Berkeley")
oak = prep_gdf(oak_parcels, "net_scale_increase", "Oakland")
sf = prep_gdf(sf_parcels, "scale_change", "San Francisco")

combined = pd.concat([berk, oak, sf], ignore_index=True)

# Lightly simplify parcel boundaries (~1m tolerance) so the HTML loads smoothly in browsers
combined["geometry"] = combined["geometry"].simplify(
    tolerance=0.00001, preserve_topology=True
)

# 3. Match the exact categories and color palette from your Oakland map
label_map = {
    0: "no change",
    1: "+1",
    2: "+2",
    3: "+3",
    4: "+4",
    5: "+5",
}
color_map = {
    "no change": "#9e9e9e",  # Grey
    "+1": "#f0f921",  # Yellow
    "+2": "#fca636",  # Orange
    "+3": "#e16462",  # Coral
    "+4": "#b12a90",  # Magenta
    "+5": "#6a00a8",  # Purple
}

combined = combined.dropna(subset=["scale_change"]).copy()
combined["points_added"] = (
    combined["scale_change"].round().astype(int).map(label_map)
)
combined = combined.dropna(subset=["points_added"])

categories = ["no change", "+1", "+2", "+3", "+4", "+5"]
combined["points_added"] = pd.Categorical(
    combined["points_added"], categories=categories, ordered=True
)
palette = [color_map[c] for c in categories]

# 4. Generate the interactive map
m = combined.explore(
    column="points_added",
    categorical=True,
    cmap=palette,
    tiles="CartoDB positron",  # Clean grey basemap matching your reference
    legend=True,
    legend_kwds={"caption": "Points added"},
    tooltip=["city", "points_added"],
    style_kwds={"weight": 0.2, "color": "white", "fillOpacity": 0.85},
)

# 5. Add a floating title banner
title_html = """
<div style="position: fixed; top: 12px; left: 50%; transform: translateX(-50%);
            z-index: 9999; background-color: rgba(255, 255, 255, 0.92);
            padding: 8px 16px; border-radius: 6px; font-family: sans-serif;
            font-size: 16px; font-weight: bold; box-shadow: 0 2px 6px rgba(0,0,0,0.2);">
    How Many Points Each Parcel Gained in Density Following SB 79
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# 6. Save to HTML
m.save("../visualizations/sb79_bay_area_scale_change.html")